# Modul 14: Vorwärtsrechnung, Backpropagation, MLPs und Faltungen

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Vorwärts und Rückwärts, MLP und Faltungen  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene NumPy-Anwendung mit mathematischem Denken  
    **Orientierungszeit:** etwa 150 bis 210 Minuten

    ## Überblick

    Sie implementieren zentrale Bausteine neuronaler Netze mit NumPy. Von einem künstlichen Neuron über stabile Softmax- und Verlustberechnungen führt der Weg zu Backpropagation, Mini-Batch-Training und einfachen CNN-Operationen.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_14A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_14B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Neuronen, Dense-Schichten und Aktivierungsfunktionen mit NumPy berechnen.
- Eine numerisch stabile Vorwärtsrechnung mit Softmax und Kreuzentropie umsetzen.
- Gradienten mit der Kettenregel herleiten und numerisch kontrollieren.
- Ein kleines MLP mit Mini-Batches, Momentum und L2-Regularisierung trainieren.
- Overfitting mit Validierungsdaten und Early Stopping erkennen.
- Faltung, Padding, Pooling und die resultierenden Tensorformen nachvollziehen.

    ## Bewertete Fähigkeiten

    - Matrixmultiplikation und Aktivierungsfunktionen
- stabile Softmax- und Kreuzentropieberechnung
- Backpropagation und numerische Gradientenprüfung
- Mini-Batch-SGD, Momentum, L2 und Early Stopping
- zweidimensionale Faltung, Padding, Max-Pooling und Formplanung

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

# Ein kleiner nichtlinearer Datensatz reicht aus, um den Nutzen einer
# verborgenen Schicht zu untersuchen. Alle Daten entstehen lokal.
X_14, y_14 = make_moons(n_samples=480, noise=0.22, random_state=RANDOM_SEED)
X_train_14, X_temp_14, y_train_14, y_temp_14 = train_test_split(
    X_14,
    y_14,
    test_size=0.40,
    stratify=y_14,
    random_state=RANDOM_SEED,
)
X_valid_14, X_test_14, y_valid_14, y_test_14 = train_test_split(
    X_temp_14,
    y_temp_14,
    test_size=0.50,
    stratify=y_temp_14,
    random_state=RANDOM_SEED,
)

# Die Skalierung wird ausschließlich mit den Trainingsdaten gelernt.
scaler_14 = StandardScaler()
X_train_14 = scaler_14.fit_transform(X_train_14)
X_valid_14 = scaler_14.transform(X_valid_14)
X_test_14 = scaler_14.transform(X_test_14)

# Ein kleines Graustufenbild und ein Kantenfilter dienen den CNN-Aufgaben.
image_14 = np.array(
    [
        [0, 0, 0, 0, 0, 0],
        [0, 1, 1, 1, 0, 0],
        [0, 1, 0, 1, 0, 0],
        [0, 1, 1, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
    ],
    dtype=np.float64,
)
vertical_edge_kernel_14 = np.array(
    [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    dtype=np.float64,
)

print("Train/Valid/Test:", X_train_14.shape, X_valid_14.shape, X_test_14.shape)

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Neuron, Dense-Schicht und Aktivierungen

    Implementieren Sie die Bausteine einer kleinen Dense-Schicht.

1. Schreiben Sie Funktionen für `sigmoid`, `tanh` und `relu`.
2. Berechnen Sie für einen einzelnen Eingabevektor zunächst den linearen Wert eines Neurons und anschließend seine Sigmoid-Ausgabe.
3. Berechnen Sie für einen Stapel aus drei Beispielen die Ausgabe einer Dense-Schicht mit drei Neuronen.
4. Wenden Sie alle drei Aktivierungsfunktionen auf dieselben Voraktivierungen an.
5. Prüfen Sie die Formen mit sinnvollen `assert`-Anweisungen und erklären Sie, warum eine reine Verkettung linearer Schichten ohne nichtlineare Aktivierung keine zusätzliche Ausdruckskraft erzeugt.

> **Hinweis:** Schreiben Sie zuerst die erwarteten Formen neben jede Matrixmultiplikation.

In [ ]:
single_input = np.array([0.8, -1.2, 0.5])
neuron_weights = np.array([0.4, -0.6, 0.2])
neuron_bias = -0.1

X_batch = np.array(
    [
        [0.8, -1.2, 0.5],
        [1.0, 0.2, -0.4],
        [-0.5, 1.5, 0.7],
    ]
)
dense_weights = np.array(
    [
        [0.3, -0.2, 0.5],
        [-0.7, 0.4, 0.1],
        [0.2, 0.6, -0.3],
    ]
)
dense_bias = np.array([0.1, -0.2, 0.05])

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Schreiben Sie zuerst die erwarteten Formen neben jede Matrixmultiplikation.

## Aufgabe 2: Stabile Vorwärtsrechnung und Kreuzentropie

    Bauen Sie eine vollständige Vorwärtsrechnung für ein zweischichtiges Mehrklassen-Netz.

1. Implementieren Sie eine zeilenweise, numerisch stabile Softmax-Funktion.
2. Berechnen Sie `X @ W1 + b1`, wenden Sie ReLU an und erzeugen Sie anschließend die Logits der zweiten Schicht.
3. Wandeln Sie die Logits in Klassenwahrscheinlichkeiten um.
4. Berechnen Sie die mittlere Kreuzentropie für ganzzahlige Klassenlabels.
5. Vergleichen Sie die stabile Softmax mit einer naiven Variante bei sehr großen Logits. Prüfen Sie, dass jede Wahrscheinlichkeitszeile ungefähr eins ergibt.

> **Hinweis:** Verwenden Sie bei allen zeilenweisen Operationen `axis=1` und `keepdims=True`.

In [ ]:
X_forward = np.array(
    [[1.0, -0.5], [0.2, 1.4], [-1.2, 0.7], [0.5, 0.3]],
    dtype=np.float64,
)
y_forward = np.array([0, 2, 1, 0])

W1 = np.array([[0.6, -0.4, 0.2], [-0.3, 0.8, 0.5]])
b1 = np.array([0.1, -0.1, 0.0])
W2 = np.array([[0.5, -0.2, 0.1], [-0.4, 0.7, 0.2], [0.3, -0.1, 0.6]])
b2 = np.array([0.0, 0.1, -0.1])

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Verwenden Sie bei allen zeilenweisen Operationen `axis=1` und `keepdims=True`.

## Aufgabe 3: Backpropagation prüfen und einen Lernschritt ausführen

    Für eine Dense-Schicht mit mittlerem quadratischem Fehler sollen Sie Vorwärts- und Rückwärtsrechnung kontrollieren.

1. Berechnen Sie Vorhersagen und MSE.
2. Leiten Sie die Gradienten nach Gewichten, Bias und Eingaben mit der Kettenregel ab.
3. Schreiben Sie eine Funktion, die den Verlust für beliebige Gewichte berechnet.
4. Prüfen Sie **alle** Gewichtselemente mit einem zentralen Differenzenquotienten und berichten Sie die größte absolute Abweichung.
5. Führen Sie einen Gradientenabstiegsschritt aus und bestätigen Sie, dass der Verlust sinkt.

> **Hinweis:** Achten Sie darauf, ob der MSE über Beispiele oder über alle Ausgabeelemente gemittelt wird.

In [ ]:
X_gradient = np.array(
    [[0.2, -0.4], [0.7, 0.1], [-0.3, 0.5], [1.0, -0.2]],
    dtype=np.float64,
)
y_gradient = np.array(
    [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]],
    dtype=np.float64,
)
W_gradient = np.array([[0.3, -0.1], [-0.2, 0.4]], dtype=np.float64)
b_gradient = np.array([[0.05, -0.05]], dtype=np.float64)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Achten Sie darauf, ob der MSE über Beispiele oder über alle Ausgabeelemente gemittelt wird.

## Aufgabe 4: Ein MLP mit Mini-Batches, Momentum und Early Stopping trainieren

    Trainieren Sie ein zweischichtiges binäres MLP auf den vorbereiteten Moon-Daten.

1. Initialisieren Sie kleine Gewichte für eine Architektur `2 -> 12 -> 1`.
2. Implementieren Sie Vorwärtsrechnung, binäre Kreuzentropie und Backpropagation.
3. Trainieren Sie mit gemischten Mini-Batches, SGD mit Momentum und L2-Regularisierung.
4. Speichern Sie pro Epoche Trainings- und Validierungsverlust.
5. Implementieren Sie Early Stopping mit Wiederherstellung der besten Parameter.
6. Berichten Sie Train-, Validierungs- und Testgenauigkeit und visualisieren Sie Verlustkurven sowie die Entscheidungsgrenze.

Verwenden Sie das Testset erst nach Abschluss aller Modellentscheidungen.

> **Hinweis:** Testen Sie die Vorwärtsrechnung zuerst auf zwei Beispielen, bevor Sie die Trainingsschleife starten.

In [ ]:
# Empfohlene Startwerte. Sie dürfen die Lernrate vorsichtig anpassen.
hidden_units = 12
max_epochs = 500
batch_size = 32
learning_rate = 0.04
momentum = 0.9
l2_strength = 0.001
patience = 45

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Testen Sie die Vorwärtsrechnung zuerst auf zwei Beispielen, bevor Sie die Trainingsschleife starten.

## Aufgabe 5: Faltung, Padding, Pooling und Tensorformen

    Implementieren und untersuchen Sie grundlegende CNN-Operationen ohne Deep-Learning-Framework.

1. Schreiben Sie eine Funktion `conv2d_single_channel(image, kernel, padding=0, stride=1)` für Kreuzkorrelation, wie sie in CNNs üblich ist.
2. Berechnen Sie die Ausgabe für das vorbereitete Bild einmal ohne Padding und einmal mit `padding=1`.
3. Schreiben Sie eine Funktion für `2 x 2` Max-Pooling mit Stride 2.
4. Vergleichen Sie berechnete und theoretisch erwartete Höhen und Breiten.
5. Planen Sie die Formen für einen Batch `(16, 28, 28, 1)` nach `Conv2D(8, 3, padding="same")`, `MaxPool2D(2)`, `Conv2D(16, 3, padding="valid")`, `MaxPool2D(2)` und anschließendem Flatten.

> **Hinweis:** Berechnen Sie zuerst nur die Ausgabehöhe. Für die Breite gilt dieselbe Formel.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Berechnen Sie zuerst nur die Ausgabehöhe. Für die Breite gilt dieselbe Formel.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.